# Read from PDF

In [ ]:
import configparser

config = configparser.ConfigParser()
config.read("config.ini")
config.sections()

contents_page = int(config["MODEL"]["ContentsPage"])
referance_no = int(config["MODEL"]["ReferanceNo"])
finantial_pages = [int(x.strip()) for x in config["MODEL"]["FinantialPages"].split(",")]

language = config["DEFAULT"]["Language"]
models_dir = config["MODEL"]["ModelDir"]
pdf_path = config["DEFAULT"]["PDFPath"]

In [ ]:
from src.pdf_reader import PDFReader

reader = PDFReader(models_dir, language)
reader.pdf_to_text_boxes(pdf_path, flush=False)

In [ ]:
from src import str_utils

print(str_utils.preprocess("Duran Varlıklar Değer_ Artış Fonundaki Değişim"))
print(str_utils.preprocess("Satışların Maliyeti (-)"))
print(str_utils.preprocess("TOPLAM VARLIKLAR"))

# Read the contents

In [ ]:
reader.read_contents(contents_page)
reader.note_pages[referance_no]

# LM split / merge texts (omitted)

# Build tables from text boxes

In [ ]:
from src.page import PageBuilder

In [ ]:
summary_page_objects = [
    PageBuilder(reader.read_summary_page(p)) for p in finantial_pages
]

page_to_summary = {}
for page, page_obj in zip(finantial_pages, summary_page_objects):
    page_to_summary[page] = page_obj.tables[0]

note = "dipnot"
for page_obj in summary_page_objects:
    page_obj.tables[0].columns = [
        note if note in str_utils.preprocess(col) else col
        for col in page_obj.tables[0].columns
    ]
    referances = page_obj.tables[0][note].map(str_utils.ocr_number_correction)
    page_obj.tables[0][note] = referances.fillna(-1).astype(int)

referance_page_objects = [
    PageBuilder(p_boxes) for p_boxes in reader.read_referance_pages(referance_no)
]

ref_page, _ = reader.note_pages[referance_no]
page_to_ref = {}
for ref_obj in referance_page_objects:
    if ref_page not in page_to_ref:
        page_to_ref[ref_page] = {}
    for ref_page_table_no, ref_df in enumerate(ref_obj.tables):
        page_to_ref[ref_page][ref_page_table_no] = ref_df
    ref_page = ref_page + 1

In [ ]:
summary_page_objects[0].paragraphs

In [ ]:
summary_page_objects[2].tables[0].head()

In [ ]:
referance_page_objects[0].paragraphs

# Build relations

## Summary Table Hierarchy

In [ ]:
df = summary_page_objects[2].tables[0].copy()
financial_cols = df.columns[2:-1]
df = df.dropna(subset=financial_cols, how="all").reset_index(drop=True)
df[financial_cols] = (
    df[financial_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
)
df

In [ ]:
row_financials = {}
parents = {}

stack, prev_l = [], int(df.iloc[0, -1])
if prev_l > 0:
    prev_l = int(df.iloc[len(df) - 1, -1])
    for i in range(len(df) - 1, -2, -1):
        level = int(df.iloc[i, -1])
        row_financials[i] = df.iloc[i, 2:-1].to_numpy()
        if len(stack):
            parents[i + 1] = stack[-1]
        if level > prev_l:
            stack.append(i + 1)
        elif level < prev_l:
            stack.pop()
        prev_l = level
else:
    for i in range(len(df)):
        level = int(df.iloc[i, -1])
        row_financials[i] = df.iloc[i, 2:-1].to_numpy()
        if len(stack):
            parents[i - 1] = stack[-1]
        if level > prev_l:
            stack.append(i - 1)
        elif level < prev_l:
            stack.pop()
        prev_l = level

childs = {}
for child, parent in parents.items():
    if parent not in childs:
        childs[parent] = []
    childs[parent].append(child)

for parent, child_list in childs.items():
    financial_sum = np.sum([row_financials[i] for i in child_list], axis=0)
    if np.array_equal(row_financials[parent], financial_sum):
        print("Good relation", parent, child_list)
    else:
        print("Broken relation", parent, child_list)

# Match summary with referance

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={"cache_dir": models_dir},
)
pairs = [
    ("brüt kar", "satış gelirleri"),
    ("brüt kar", "satışların maliyeti"),
    ("satış gelirleri", "satışların maliyeti"),
    ("brüt kar", "dönem karı dağılımı"),
]
cross_encoder.predict(pairs)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer(
    "intfloat/multilingual-e5-small", model_kwargs={"cache_dir": models_dir}
)

texts = ["brüt kar", "satış gelirleri", "satışların maliyeti", "dönem karı dağılımı"]

embeddings = model.encode([f"query: {t}" for t in texts], normalize_embeddings=True)

similarity = cosine_similarity(embeddings, embeddings)

print(similarity)

In [ ]:
def get_probabilities(query, text_list):
    embeddings1 = model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
    )
    embeddings2 = model.encode(
        [f"query: {t}" for t in text_list],
        normalize_embeddings=True,
    )

    scores = cosine_similarity(embeddings1, embeddings2)[0]
    probs = torch.softmax(torch.tensor(scores), dim=-1)
    return probs

In [ ]:
# all the rows from the referace
ref_page, _ = reader.note_pages[referance_no]
search_rows, search_cols = [], []
for ref_obj in referance_page_objects:
    for ref_page_table_no, ref_df in enumerate(ref_obj.tables):
        for row, row_j in zip(ref_df.Kalem, ref_df.index):
            original = str(row)
            row_p = str_utils.preprocess(original)
            search_rows.append([ref_page, ref_page_table_no, row_j, row_p])

        for col_j in range(1, len(ref_df.columns) - 1):
            col_p = str_utils.preprocess(str(ref_df.columns[col_j]))
            search_cols.append([ref_page, ref_page_table_no, col_j, col_p])
    ref_page = ref_page + 1

search_rows_df = pd.DataFrame(
    search_rows, columns=["Page", "Table", "RefRow", "Compare"]
)
search_cols_df = pd.DataFrame(
    search_cols, columns=["Page", "Table", "RefCol", "Compare"]
)

In [ ]:
import torch

temperature = 2

row = "yatırım amaçlı gayrimenkuller"


def get_probabilities(query, text_list):
    scores = cross_encoder.predict([(query, x) for x in text_list])
    probs = torch.softmax(torch.tensor(scores / temperature), dim=-1)
    return probs


score_df = search_rows_df.copy()
score_df["score"] = get_probabilities(row, search_rows_df.Compare.to_list())
score_df.sort_values(by="score", inplace=True, ascending=False)
score_df.head()

In [ ]:
from dataclasses import dataclass


@dataclass
class CellMatch:
    s_page: str = None
    s_row: int = None
    s_col: int = None
    r_page: str = None
    r_table: int = None
    r_row: int = None
    r_col: int = None
    score: float = None


referance_matches = []
for page, page_obj in zip(finantial_pages, summary_page_objects):
    df = page_obj.tables[0].copy()
    # rows containing the reference
    df = df[df[note] == referance_no]
    if len(df) == 0:
        continue

    # row by row similarity
    for row, row_i in zip(df.Kalem, df.index):
        row = str_utils.preprocess(str(row))
        text_list = search_rows_df.Compare.to_list()
        row_scores = get_probabilities(row, text_list)

        # column by row similarity
        for col_i in range(2, len(df.columns) - 1):
            col = str_utils.preprocess(str(df.columns[col_i]))
            col_scores = get_probabilities(col, text_list)

            # combine probs
            score_df = search_rows_df.copy()
            score = row_scores * col_scores
            score_df["score"] = score
            score_df.sort_values(by="score", inplace=True, ascending=False)

            # select top table rows
            top = next(score_df.itertuples())
            referance_matches.append(
                CellMatch(
                    s_page=page,
                    s_row=row_i,
                    s_col=col_i,
                    r_page=top.Page,
                    r_table=top.Table,
                    r_row=top.RefRow,
                    score=top.score,
                )
            )

    # column by column similarity
    for _match in referance_matches:
        score_df = search_cols_df[
            (search_cols_df["Page"] == _match.r_page)
            & (search_cols_df["Table"] == _match.r_table)
        ].copy()
        col = str_utils.preprocess(str(df.columns[_match.s_col]))
        text_list = score_df.Compare.to_list()
        score_df["score"] = get_probabilities(col, text_list)
        score_df.sort_values(by=["score", "RefCol"], inplace=True, ascending=False)
        top = next(score_df.itertuples())
        _match.r_col = top.RefCol
        _match.score = _match.score * top.score

referance_matches

In [ ]:
score_df

In [ ]:
for _match in referance_matches:
    s_value = page_to_summary[_match.s_page].iloc[_match.s_row, _match.s_col]
    r_value = page_to_ref[_match.r_page][_match.r_table].iloc[
        _match.r_row, _match.r_col
    ]
    print(s_value, r_value)
    print(s_value == r_value)

# NER

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

model_path = "akdeniz27/bert-base-turkish-cased-ner"
tokenizer = AutoTokenizer.from_pretrained(model_path, cache_dir=models_dir)
model = AutoModelForTokenClassification.from_pretrained(
    model_path, cache_dir=models_dir
)
ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first",
)

In [ ]:
text = (
    "Standart Gayrimenkul Değerlendirme Uygulamaları A.Ş., "
    "Sermaye Piyasası Kurulu tarafından yetkilendirilmiş "
    "bir gayrimenkul değerleme şiketidir"
)

spans = []
for entity in ner(text):
    if entity["entity_group"] != "ORG":
        continue
    start, end, score = entity["start"], entity["end"], float(entity["score"])
    if spans and start <= spans[-1][1] + 1:
        last = spans.pop()
        start, score = last[0], min(last[2], score)
    spans.append((start, end, score))
[(text[a:b].strip(), score) for a, b, score in spans]